# RAG-ассистент по документации для малого бизнеса · Qwen · оценка качества

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/serge3emskov/rag-smb-assistant/blob/main/notebooks/rag_smb_assistant_colab.ipynb)

Прототип корпоративного AI-ассистента: от сырых HTML/PDF до измеренного качества.

| Этап | Что делаем |
|---|---|
| 1. Данные | HTML/PDF → очистка → разделы по заголовкам → чанки с путём заголовков |
| 2. Поиск | BM25 + dense (multilingual-e5) → RRF → cross-encoder реранкер |
| 3. Сущности | каталог тарифов и продуктов, падежи и опечатки, буст «своих» разделов |
| 4. Генерация | Qwen3-4B-Instruct, ответ только по контексту, ссылки [n], фиксированный отказ |
| 5. Оценка | golden dataset, Hit@k / MRR / Recall, fact recall, галлюцинации, LLM-судья, классификация ошибок |

**Среда:** `Среда выполнения → Сменить среду выполнения → T4 GPU`. Полный прогон занимает ~10 минут.

## 1. Установка

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/serge3emskov/rag-smb-assistant"   # ← ваш репозиторий
WORKDIR = "/content/rag-smb-assistant"

if not os.path.exists(WORKDIR):
    if "<GITHUB_USER>" not in REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, WORKDIR], check=True)
    else:
        # репозиторий ещё не опубликован — загрузите zip проекта
        from google.colab import files
        up = files.upload()
        zname = next(iter(up))
        subprocess.run(["unzip", "-q", zname, "-d", "/content"], check=True)
else:
    subprocess.run(["git", "-C", WORKDIR, "pull", "-q"])     # репозиторий уже есть — подтянуть обновления
os.chdir(WORKDIR)
!pip install -q -e ".[gpu,api,dev]"
!pip install -q -U "transformers>=4.51"
!nvidia-smi --query-gpu=name,memory.total --format=csv

# pip install -e пишет .pth-файл, который Python читает только при старте ядра,
# поэтому без перезапуска среды пакет не импортируется — добавляем src в путь вручную
import sys, importlib
sys.path.insert(0, os.path.join(WORKDIR, "src"))
importlib.invalidate_caches()
import smbrag; print("smbrag", smbrag.__version__)

## 2. Корпус

* `DEMO` — документация вымышленного «Банка Пример» (7 документов, 42 вопроса), работает сразу.
* `SBER` — страницы СберБизнеса, сохранённые из браузера (см. `data/sber/README.md`). Загрузите HTML/PDF в ячейке ниже.

In [ ]:
CORPUS = "DEMO"          # "DEMO" | "SBER"

import yaml, pathlib
cfg = yaml.safe_load(open("configs/colab.yaml", encoding="utf-8"))
if CORPUS == "SBER":
    cfg["docs_dir"] = "data/sber/docs"
    cfg["entities"]["catalog"] = "data/sber/entities.yaml"
    GOLDEN = "eval/golden_sber.jsonl"
    docs_dir = pathlib.Path(cfg["docs_dir"])
    if not any(p.suffix.lower() in (".html", ".htm", ".pdf", ".md", ".txt") for p in docs_dir.glob("*")):
        from google.colab import files
        for name, data in files.upload().items():
            (docs_dir / name).write_bytes(data)
else:
    GOLDEN = "eval/golden_demo.jsonl"
print(cfg["docs_dir"], "|", GOLDEN)

### Документы и чанки
Смотрим, что реально попадает в индекс: мусор из меню и cookie-баннеров — первая причина плохого поиска.

In [ ]:
import pandas as pd
from smbrag.loaders import load_dir
from smbrag.chunking import chunk_documents

docs = load_dir(cfg["docs_dir"])
chunks = chunk_documents(docs, **cfg["chunking"])
print(f"Документов: {len(docs)}, разделов: {sum(len(d.sections) for d in docs)}, чанков: {len(chunks)}")

lens = pd.Series([len(c.text) for c in chunks])
display(pd.DataFrame({"doc_id": [c.doc_id for c in chunks], "heading": [c.heading for c in chunks], "chars": lens}).head(12))
print(lens.describe().round(0).to_dict())

## 3. Сопоставление сущностей
Каталог `entities.yaml` + стемминг (падежи) + fuzzy (опечатки). Чанк помечается сущностями из текста и из заголовка раздела.

In [ ]:
from smbrag.entities import EntityCatalog, suggest_entities

catalog = EntityCatalog.from_yaml(cfg["entities"]["catalog"], fuzzy_threshold=cfg["entities"]["fuzzy_threshold"])
catalog.tag_chunks(chunks)
print("Сущностей в каталоге:", len(catalog.entities))

if CORPUS == "SBER":
    print("Черновик каталога (названия в кавычках из документов):")
    display(pd.DataFrame(suggest_entities(chunks), columns=["название", "упоминаний"]).head(30))

for q in ["Сколько платежей бесплатно на Деловом ритме?", "Сколько стоит тариф Делавой ритм?",
          "Под какой процент кредит без залога?", "Какие документы нужны для открытия счёта?"]:
    print(f"{q!r:60} →", [(m.entity.name, round(m.score)) for m in catalog.match(q)])

### Golden dataset для СберБизнеса

Если `eval/golden_sber.jsonl` ещё нет, Qwen составит черновик по вашим страницам: вопрос словами клиента,
дословная цитата-обоснование и ключевой факт. Вопросы, цитату которых не удалось найти в тексте, отбрасываются.

**Черновик нужно просмотреть:** удалить тривиальные вопросы, проверить факты, добавить вопросы про путаницу похожих
продуктов и multi-hop, проверить, что «неотвечаемых» вопросов действительно нет на страницах. Затем скачать файл
и положить в репозиторий как `eval/golden_sber.jsonl`.

In [ ]:
import os
from smbrag.llm import build_generator

gen = globals().get("gen")
DRAFT = "eval/golden_sber_draft.jsonl"
if CORPUS == "SBER" and not os.path.exists(GOLDEN) and os.path.exists(DRAFT):
    GOLDEN = DRAFT                                                 # черновик уже есть — не генерируем заново
    print("Используется существующий черновик:", DRAFT)
elif CORPUS == "SBER" and not os.path.exists(GOLDEN):
    from smbrag.synth import generate_golden_draft, unanswerable_rows, write_jsonl
    gen = gen or build_generator(cfg["generation"])                # Qwen загружается один раз и переиспользуется ниже
    rows, stats = generate_golden_draft(chunks, gen, n=N_QUESTIONS if "N_QUESTIONS" in dir() else 40)
    rows += unanswerable_rows(len(rows) + 1)
    GOLDEN = DRAFT
    write_jsonl(rows, GOLDEN, header="Черновик, сгенерированный Qwen по сохранённым страницам. Проверить вручную.")
    print("Статистика генерации:", stats)
    print(f"Сохранено: {GOLDEN} ({len(rows)} вопросов)")
    display(pd.DataFrame(rows)[["id", "question", "expected_docs", "evidence", "expected_facts"]].head(20))
    # from google.colab import files; files.download(GOLDEN)

## 4. Поиск: сравнение конфигураций
Метрики только по ретриверу — без генерации, поэтому быстро. Dense и реранкер загружаются на GPU.

In [ ]:
from smbrag.golden import load_golden, validate_golden
from smbrag.cli import retrieval_ablation

items = load_golden(GOLDEN)
problems = validate_golden(items, chunks)
print(f"Вопросов: {len(items)}; проблем разметки: {len(problems)}")
for p in problems[:10]: print("  ", p)

abl = pd.DataFrame(retrieval_ablation(cfg, items, k=cfg["retrieval"]["k"])).set_index("config")
display(abl.style.format("{:.1%}").background_gradient(cmap="Greens", axis=0))

In [ ]:
import matplotlib.pyplot as plt
ax = abl[["hit@1", "mrr"]].plot.barh(figsize=(8, 3.5), color=["#2a78b5", "#8fbce0"])
ax.set_xlim(0.5, 1); ax.set_xlabel(""); ax.invert_yaxis(); ax.legend(loc="lower right")
ax.set_title("Качество поиска по конфигурациям"); plt.tight_layout(); plt.show()

## 5. Qwen: генерация с цитированием

In [ ]:
from smbrag.pipeline import Assistant

asst = Assistant.from_config(cfg, generator=gen)   # hybrid(bm25+dense) → rerank → entities → Qwen (gen — если уже загружен)
print("Ретривер:", asst.retriever.name, "| модель:", asst.generator.name)

for q in ["Сколько стоит обслуживание на тарифе Первый шаг?",
          "Какая ставка за оплату по QR-коду для салона красоты?",
          "Какая ставка по ипотеке для ИП?"]:
    print(asst.ask(q).pretty(), end="\n\n")

## 6. Массовый прогон golden dataset

Для каждого вопроса: поиск → ответ Qwen → rule-based проверка (числа и ссылки) → LLM-судья по утверждениям →
классификация ошибки по первому сломавшемуся этапу.

> Судья — та же Qwen, что и генератор. Это ускоряет прогон, но судья склонен прощать собственные ошибки;
> для итоговых цифр возьмите другую модель (например, Qwen3-8B в 4-bit) или проверьте выборку вручную.

In [ ]:
from smbrag.evaluation import evaluate_end_to_end, summarize, breakdown, write_report

results = evaluate_end_to_end(asst, items)
summary = summarize(results)
pd.Series({k: v for k, v in summary.items() if k != "errors"}).to_frame("Qwen")

In [ ]:
err = pd.Series(summary["errors"])
ax = err.plot.barh(figsize=(7, 3), color="#c0504d"); ax.invert_yaxis(); ax.set_title("Результат по вопросам")
plt.tight_layout(); plt.show()
display(pd.DataFrame(breakdown(results, "difficulty")).T.style.format({"accuracy": "{:.0%}", "fact_recall": "{:.0%}", "hallucination_rate": "{:.0%}"}))

### Разбор ошибок
Где именно сломалось и что чинить — подсказка из `ERROR_HINTS`.

In [ ]:
from smbrag.evaluation import ERROR_HINTS
bad = pd.DataFrame([{
    "id": r.id, "ошибка": r.error, "вопрос": r.question, "ответ": r.answer[:160],
    "ранг релевантного": r.first_relevant_rank, "числа без опоры": r.unsupported_numbers,
    "что чинить": ERROR_HINTS.get(r.error, ""),
} for r in results if not r.correct])
pd.set_option("display.max_colwidth", 160)
display(bad)

## 7. Что даёт LLM: сравнение с экстрактивным baseline
Тот же ретривер, вместо Qwen — выбор предложений из контекста.

In [ ]:
from smbrag.llm import ExtractiveBaseline
base = Assistant(asst.chunks, asst.retriever, ExtractiveBaseline(), asst.k)
base_summary = summarize(evaluate_end_to_end(base, items, progress=False))
cols = ["accuracy", "answerable_accuracy", "fact_recall", "hallucination_rate", "false_refusal_rate", "refusal_accuracy_unanswerable"]
cmp_ = pd.DataFrame({"extractive": {k: base_summary[k] for k in cols}, "Qwen": {k: summary[k] for k in cols}})
display(cmp_.style.format("{:.1%}"))

## 8. Отчёт

In [ ]:
out = f"reports/{CORPUS.lower()}_qwen"
path = write_report(results, out, title=f"RAG-ассистент · {CORPUS} · {asst.generator.name}",
                    meta={"retriever": asst.retriever.name, "generator": asst.generator.name, "k": asst.k, "chunks": len(asst.chunks)},
                    ablation=abl.reset_index().to_dict("records"))
print(open(path, encoding="utf-8").read()[:3000])
# from google.colab import files; files.download(path)

## 9. Спросить ассистента

In [ ]:
question = "Сколько стоит аренда терминала и когда её не берут?"  # @param {type:"string"}
print(asst.ask(question).pretty())